# UWPR Contaminant Peak Analysis in NIST20 Data

Two-stage analysis:
1. **Stage 1 (Mass match)**: Find NIST20 peaks whose m/z matches a known UWPR contaminant within tolerance
2. **Stage 2 (Formula feasibility)**: Among mass-matched peaks, check if the contaminant formula could be a real fragment of the parent molecule (all element counts ≤ parent formula → plausibly real peak)

Peaks that match by mass but are **formula-infeasible** (contaminant formula has more of some element than the parent) are strong contaminant candidates.

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

## 1. Load UWPR / MaConDa contaminant lists

In [2]:
DATA_DIR = Path("../data/umpr")

# --- MaConDa (has formula + mz + ion_mode + type) ---
maconda = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(MaConDa).csv",
    skiprows=3,
    dtype=str,
    encoding="latin-1",
)
maconda.columns = maconda.columns.str.strip()
maconda["mz"] = pd.to_numeric(maconda["mz"], errors="coerce")
maconda["exact_adduct_mass"] = pd.to_numeric(maconda["exact_adduct_mass"], errors="coerce")
maconda = maconda.dropna(subset=["mz"])
maconda["ion_mode"] = maconda["ion_mode"].str.strip()
print(f"MaConDa entries: {len(maconda)}, ion modes: {maconda['ion_mode'].value_counts().to_dict()}")
maconda.head(3)

MaConDa entries: 564, ion modes: {'POS': 545, 'NEG': 19}


,name,formula,exact_mass,std_inchi,std_inchi_key,id,type_of_contaminant,pubchem_cid,reference,ion_mode,mz,exact_adduct_mass,ion_form,instrument_type,instrument,chromatography,ion_source_type
0,Acetic Acid,C2H4O2,60.02113,"InChI=1S/C2H4O2/c1-2(3)4/h1H3,(H,3,4)",QTBSBXVTEAMEQO-UHFFFAOYSA-N,CON00001,Solvent,176,"Tong, H., Bell, D., Tabei, K., & Siegel, M. M....",NEG,59.00,59.013854,[M-H]-,Ion trap,Micromass Platform II,LC,ESI
1,Acetic Acid,C2H4O2,60.02113,"InChI=1S/C2H4O2/c1-2(3)4/h1H3,(H,3,4)",QTBSBXVTEAMEQO-UHFFFAOYSA-N,CON00001,Solvent,176,"Ijames, C. F., Dutky, R. C., & Fales, H. M. (1...",POS,537.88,537.879013,[M6-H6+Fe3+O]+,Triple quadrupole,Finnigan TSQ-700,NaN,ESI
2,Acetic Acid,C2H4O2,60.02113,"InChI=1S/C2H4O2/c1-2(3)4/h1H3,(H,3,4)",QTBSBXVTEAMEQO-UHFFFAOYSA-N,CON00001,Solvent,176,"Ijames, C. F., Dutky, R. C., & Fales, H. M. (1...",POS,555.00,555.889578,[M6-H6+H2O+Fe3+O]+,Triple quadrupole,Finnigan TSQ-700,NaN,ESI


In [3]:
# --- UWPR Positive file (Keller et al. 2008) ---
uwpr_pos_raw = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(Positive).csv",
    skiprows=7,
    header=None,
    dtype=str,
    encoding="latin-1",
)
uwpr_pos = uwpr_pos_raw[[0, 1, 2, 3, 4]].copy()
uwpr_pos.columns = ["mz", "ion_type", "formula", "compound", "origin"]
uwpr_pos["mz"] = pd.to_numeric(uwpr_pos["mz"], errors="coerce")
uwpr_pos = uwpr_pos.dropna(subset=["mz"])
uwpr_pos["ion_mode"] = "POS"
print(f"UWPR Positive entries: {len(uwpr_pos)}")
uwpr_pos.head(3)

UWPR Positive entries: 822


,mz,ion_type,formula,compound,origin,ion_mode
1,33.033491,[M+H]+,CH3OH,Methanol,"Acetonitrile, solvent",POS
2,42.033825,[M+H]+,CH3CN,ACN,"Acetonitrile, solvent",POS
3,59.060374,[M+NH4]+,CH3CN,ACN,"Acetonitrile, solvent",POS


In [4]:
# --- UWPR Negative file ---
uwpr_neg_raw = pd.read_csv(
    DATA_DIR / "UWPR_CommonMassSpecContaminants(Negative).csv",
    skiprows=7,
    header=None,
    dtype=str,
    encoding="latin-1",
)
uwpr_neg = uwpr_neg_raw[[0, 1, 2, 3, 4]].copy()
uwpr_neg.columns = ["mz", "ion_type", "formula", "compound", "origin"]
uwpr_neg["mz"] = pd.to_numeric(uwpr_neg["mz"], errors="coerce")
uwpr_neg = uwpr_neg.dropna(subset=["mz"])
uwpr_neg["ion_mode"] = "NEG"
print(f"UWPR Negative entries: {len(uwpr_neg)}")
uwpr_neg.head(3)

UWPR Negative entries: 28


,mz,ion_type,formula,compound,origin,ion_mode
2,26.00362,f-,CN,CN-,Fragment from acetonitrile,NEG
3,44.99820,[M-H]-,HCOOH,(HCOO)-,"Formic acid, FA (anion), Formate",NEG
4,59.01385,[M-H]-,CH3COOH,(CH3CO2)-,"Acetic Acid, Ac (anion), Acetate",NEG


In [5]:
MZ_MAX = 1500.0  # consistent with spec_utils.MZ_MAX

# Build unified contaminant tables keyed by ion_mode
# MaConDa as primary (has formula info); UWPR Keller list fills in extra m/z values
maconda_pos = maconda[maconda["ion_mode"] == "POS"][
    ["mz", "formula", "name", "type_of_contaminant", "ion_form"]
].copy()
maconda_pos.columns = ["mz", "formula", "compound", "type", "ion_form"]
maconda_pos["source"] = "MaConDa"

maconda_neg = maconda[maconda["ion_mode"] == "NEG"][
    ["mz", "formula", "name", "type_of_contaminant", "ion_form"]
].copy()
maconda_neg.columns = ["mz", "formula", "compound", "type", "ion_form"]
maconda_neg["source"] = "MaConDa"

# UWPR Keller pos
uwpr_pos_slim = uwpr_pos[["mz", "formula", "compound", "ion_type"]].copy()
uwpr_pos_slim["type"] = "unknown"
uwpr_pos_slim["ion_form"] = uwpr_pos_slim["ion_type"]
uwpr_pos_slim["source"] = "UWPR_Keller"
uwpr_pos_slim = uwpr_pos_slim.drop(columns=["ion_type"])

# Merge, deduplicate, and drop entries above MZ_MAX
cont_pos = (
    pd.concat([maconda_pos, uwpr_pos_slim], ignore_index=True)
    .drop_duplicates(subset=["mz"])
    .query("mz <= @MZ_MAX")
    .reset_index(drop=True)
)
# UWPR Keller neg
uwpr_neg_slim = uwpr_neg[["mz", "formula", "compound", "ion_type"]].copy()
uwpr_neg_slim["type"] = "unknown"
uwpr_neg_slim["ion_form"] = uwpr_neg_slim["ion_type"]
uwpr_neg_slim["source"] = "UWPR_Keller"
uwpr_neg_slim = uwpr_neg_slim.drop(columns=["ion_type"])

cont_neg = (
    pd.concat([maconda_neg, uwpr_neg_slim], ignore_index=True)
    .drop_duplicates(subset=["mz"])
    .query("mz <= @MZ_MAX")
    .reset_index(drop=True)
)

print(f"Unique contaminant m/z values — POS: {len(cont_pos)}, NEG: {len(cont_neg)}")

Unique contaminant m/z values — POS: 1100, NEG: 44


## 2. Load NIST23 processed data

In [6]:
PROC_DIR = Path("../data/proc/nist20")

spec_df = pd.read_pickle(PROC_DIR / "spec_df.pkl")
mol_df = pd.read_pickle(PROC_DIR / "mol_df.pkl")

print(f"spec_df: {len(spec_df)} spectra, columns: {list(spec_df.columns)}")
print(f"mol_df:  {len(mol_df)} molecules, columns: {list(mol_df.columns)}")

spec_df: 1148072 spectra, columns: ['spec_id', 'mol_id', 'prec_type', 'inst_type', 'frag_mode', 'spec_type', 'ion_mode', 'dset', 'dset_spec_id', 'col_gas', 'res', 'ace', 'nce', 'prec_mz', 'peaks', 'ri', 'group_id', 'ace_extra_1', 'nce_extra_1', 'ace_extra_2', 'nce_extra_2', 'ce_type']
mol_df:  35025 molecules, columns: ['smiles', 'mol_id', 'mol', 'inchikey_s', 'scaffold', 'formula', 'inchi', 'mw', 'exact_mw', 'num_atoms', 'num_bonds', 'charge', 'single_mol', 'num_radicals']


In [7]:
# Merge formula into spec_df
spec_mol = spec_df.merge(mol_df[["mol_id", "formula"]], on="mol_id", how="left")
print(f"ion_mode distribution: {spec_mol['ion_mode'].value_counts().to_dict()}")
spec_mol.head(2)

ion_mode distribution: {'P': 846400, 'N': 301662, 'E': 8}


,spec_id,mol_id,prec_type,inst_type,frag_mode,spec_type,ion_mode,dset,dset_spec_id,col_gas,...,prec_mz,peaks,ri,group_id,ace_extra_1,nce_extra_1,ace_extra_2,nce_extra_2,ce_type,formula
0,0,10998,[M+NH4]+,QTOF,CID,MS2,P,nist20_hr,1035166,N2,...,1467.0061,"[(45.96, 46.35), (597.48, 51.85), (598.5, 41.3...",NaN,0,NaN,NaN,NaN,NaN,single,C81H142O17P2
1,1,10998,[M+NH4]+,QTOF,CID,MS2,P,nist20_hr,1035167,N2,...,1467.0061,"[(597.5, 50.95), (599.5, 683.32), (1449.98, 99...",NaN,0,NaN,NaN,NaN,NaN,single,C81H142O17P2


## 3. Parse peaks into (mz, intensity) arrays

Peaks are stored as Python list-of-tuples literals: `[(mz, intensity), ...]`

In [8]:
import re


def parse_formula(formula: str) -> dict[str, int]:
    """Parse a molecular formula string into an element count dict.

    Args:
        formula: Molecular formula string, e.g. "C6H12O6".

    Returns:
        Dict mapping element symbol to count, e.g. {"C": 6, "H": 12, "O": 6}.

    Raises:
        ValueError: If formula is empty or unparseable.
    """
    if not formula or not isinstance(formula, str):
        raise ValueError(f"Invalid formula: {formula!r}")
    counts: dict[str, int] = {}
    for elem, num in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        if elem:
            counts[elem] = counts.get(elem, 0) + (int(num) if num else 1)
    if not counts:
        raise ValueError(f"No elements parsed from: {formula!r}")
    return counts


def parse_peaks(peaks) -> np.ndarray:
    """Parse peaks stored as list of (mz, intensity) tuples into (N, 2) float32 array."""
    if peaks is None:
        return np.empty((0, 2), dtype=np.float32)
    if isinstance(peaks, str):
        try:
            peaks = ast.literal_eval(peaks)
        except Exception:
            return np.empty((0, 2), dtype=np.float32)
    if not peaks:
        return np.empty((0, 2), dtype=np.float32)
    return np.array(peaks, dtype=np.float32)


# Quick sanity check
sample = spec_mol["peaks"].iloc[0]
arr = parse_peaks(sample)
print(f"Sample peaks type: {type(sample).__name__}, parsed shape: {arr.shape}")
print(f"First 3 peaks: {arr[:3]}")
assert parse_formula("C6H12O6") == {"C": 6, "H": 12, "O": 6}
print("parse_formula OK")


Sample peaks type: list, parsed shape: (15, 2)
First 3 peaks: [[ 45.96  46.35]
 [597.48  51.85]
 [598.5   41.36]]
parse_formula OK


In [9]:
# Build spec_pn as a metadata-only table (no peaks column) for summary stats later.
# Peaks are processed on-the-fly per ion mode in the matching step to avoid holding
# the full exploded peak table (~57M rows) in memory all at once.
META_COLS = ["spec_id", "mol_id", "formula", "ion_mode", "inst_type"]
spec_pn = spec_mol[spec_mol["ion_mode"].isin(["P", "N"])][META_COLS].copy()
print(f"spec_pn (metadata): {len(spec_pn):,} spectra")

spec_pn (metadata): 1,148,062 spectra


## 4. Formula feasibility check

In [10]:
def formula_feasible(parent_formula: str, contaminant_formula: str) -> bool | None:
    """Return True if contaminant formula could be a subfragment of parent.

    A contaminant formula is feasible as a real fragment if every element count
    in the contaminant formula is <= the corresponding count in the parent.
    Allows ±4 H flexibility to account for rearrangements.

    Returns None if either formula cannot be parsed.
    """
    H_FLEX = 4
    try:
        p = parse_formula(parent_formula)
        c = parse_formula(contaminant_formula)
    except Exception:
        return None

    for elem, count in c.items():
        if elem == "H":
            if count > p.get("H", 0) + H_FLEX:
                return False
        else:
            if count > p.get(elem, 0):
                return False
    return True


assert formula_feasible("C6H12O6", "C2H4O") is True   # glucose → ethylene oxide fragment
assert formula_feasible("C6H12O6", "C7H6O") is False  # more C than parent
print("formula_feasible OK")

formula_feasible OK


## 5. Stage 1: Mass-match peaks vs. UWPR list (vectorised)

Uses `pd.merge_asof` with `direction='nearest'` and `tolerance` for O(N log N) matching.

In [11]:
# Instrument-type-specific tolerances
# FT (Orbitrap): 10 ppm, QTOF: 20 ppm, IT: 0.3 Da, QQQ/Other: 0.1 Da
INST_TOL: dict[str, tuple[str, float]] = {
    "FT":   ("ppm", 10.0),
    "QTOF": ("ppm", 20.0),
    "IT":   ("da",  0.30),
    "QQQ":  ("da",  0.10),
}
DEFAULT_TOL: tuple[str, float] = ("da", 0.05)


def tol_da_for(inst_type: str, mz: float) -> float:
    """Return absolute Da tolerance for a given instrument type and m/z."""
    mode, val = INST_TOL.get(inst_type, DEFAULT_TOL)
    return mz * val * 1e-6 if mode == "ppm" else val


cont_pos_sorted = cont_pos.sort_values("mz").reset_index(drop=True)
cont_neg_sorted = cont_neg.sort_values("mz").reset_index(drop=True)

CONT_RENAME = {
    "mz": "cont_mz", "formula": "cont_formula", "compound": "cont_compound",
    "type": "cont_type", "ion_form": "cont_ion_form", "source": "cont_source",
}


def _match_group(group: pd.DataFrame, cont_sorted: pd.DataFrame) -> pd.DataFrame:
    """Run merge_asof for one inst_type group, then post-filter by exact per-peak ppm/Da."""
    inst = group["inst_type"].iloc[0]
    mode, val = INST_TOL.get(inst, DEFAULT_TOL)
    # scalar upper-bound tolerance for merge_asof (worst case: highest m/z in group)
    max_mz = group["peak_mz"].max()
    scalar_tol = max_mz * val * 1e-6 if mode == "ppm" else val

    right = cont_sorted[["mz", "formula", "compound", "type", "ion_form", "source"]].rename(
        columns=CONT_RENAME
    )
    merged = pd.merge_asof(
        group.sort_values("peak_mz"), right,
        left_on="peak_mz", right_on="cont_mz",
        direction="nearest", tolerance=scalar_tol,
    )
    hits = merged.dropna(subset=["cont_mz"]).copy()
    hits["delta_mz"] = hits["peak_mz"] - hits["cont_mz"]
    # exact per-peak filter: |delta| <= tol(inst, peak_mz)
    exact_tol = hits["peak_mz"].map(lambda mz: tol_da_for(inst, mz))
    hits = hits[hits["delta_mz"].abs() <= exact_tol].copy()
    hits["tol_da"] = exact_tol[hits.index]
    return hits


def explode_and_match(
    spec_with_peaks: pd.DataFrame, cont_sorted: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parse and explode peaks for one ion mode, match vs contaminant list.

    Uses per-instrument-type tolerances (ppm for FT/QTOF, Da for IT/QQQ).

    Returns:
        hits: DataFrame of matched (peak, contaminant) pairs.
        totals: Per-spectrum total_peaks and total_intensity.
    """
    slim = spec_with_peaks[META_COLS + ["peaks"]].copy()
    slim["peak_mz"] = slim["peaks"].progress_map(
        lambda p: parse_peaks(p)[:, 0].tolist() if parse_peaks(p).shape[0] > 0 else []
    )
    slim["peak_int"] = slim["peaks"].map(
        lambda p: parse_peaks(p)[:, 1].tolist() if parse_peaks(p).shape[0] > 0 else []
    )
    slim = slim.drop(columns=["peaks"])

    flat = slim.explode(["peak_mz", "peak_int"]).dropna(subset=["peak_mz"])
    flat["peak_mz"] = flat["peak_mz"].astype(np.float64)
    flat["peak_int"] = flat["peak_int"].astype(np.float64)
    flat = flat[flat["peak_mz"] <= MZ_MAX]
    del slim

    totals = (
        flat.groupby("spec_id")
        .agg(total_peaks=("peak_mz", "count"), total_intensity=("peak_int", "sum"))
        .reset_index()
    )

    hits = pd.concat(
        [_match_group(grp, cont_sorted) for _, grp in flat.groupby("inst_type", observed=True)],
        ignore_index=True,
    )
    del flat
    return hits, totals


print("Matching positive mode peaks...")
hits_pos, totals_pos = explode_and_match(
    spec_mol[spec_mol["ion_mode"] == "P"], cont_pos_sorted
)
print(f"  POS hits: {len(hits_pos):,} in {hits_pos['spec_id'].nunique():,} spectra")

print("Matching negative mode peaks...")
hits_neg, totals_neg = explode_and_match(
    spec_mol[spec_mol["ion_mode"] == "N"], cont_neg_sorted
)
print(f"  NEG hits: {len(hits_neg):,} in {hits_neg['spec_id'].nunique():,} spectra")

hits_df = pd.concat([hits_pos, hits_neg], ignore_index=True)
totals_df = pd.concat([totals_pos, totals_neg], ignore_index=True)
del hits_pos, hits_neg, totals_pos, totals_neg
print(f"\nTotal hit (peak, contaminant) pairs: {len(hits_df):,}")
print(f"Unique spectra with ≥1 mass hit:     {hits_df['spec_id'].nunique():,}")
print(f"Unique contaminant m/z matched:      {hits_df['cont_mz'].nunique():,}")
print(f"\nHits by instrument type:")
print(hits_df.groupby("inst_type")[["spec_id"]].nunique().rename(columns={"spec_id": "n_spectra"}).to_string())


Matching positive mode peaks...


  0%|          | 0/846400 [00:00<?, ?it/s]

  POS hits: 1,050,164 in 265,922 spectra
Matching negative mode peaks...


  0%|          | 0/301662 [00:00<?, ?it/s]

  NEG hits: 123,762 in 79,525 spectra

Total hit (peak, contaminant) pairs: 1,173,926
Unique spectra with ≥1 mass hit:     345,447
Unique contaminant m/z matched:      1,141

Hits by instrument type:
           n_spectra
inst_type           
FT            293585
IT              4187
Other          11062
QQQ             1305
QTOF           35308


## 6. Stage 2: Formula feasibility filter

In [12]:
def check_feasibility(row) -> str:
    parent_f = row["formula"]
    cont_f = row["cont_formula"]
    if not isinstance(cont_f, str) or not cont_f.strip() or cont_f.strip() == "nan":
        return "no_cont_formula"
    if not isinstance(parent_f, str) or not parent_f.strip():
        return "no_parent_formula"
    result = formula_feasible(parent_f, cont_f)
    if result is None:
        return "parse_error"
    return "feasible" if result else "infeasible"


hits_df["feasibility"] = hits_df.progress_apply(check_feasibility, axis=1)
print(hits_df["feasibility"].value_counts())

  0%|          | 0/1173926 [00:00<?, ?it/s]

feasibility
feasible           648107
infeasible         508531
no_cont_formula     10324
parse_error          6964
Name: count, dtype: int64


## 7. Summary statistics

In [13]:
total_spectra = len(spec_pn)
total_pos = (spec_pn["ion_mode"] == "P").sum()
total_neg = (spec_pn["ion_mode"] == "N").sum()

n_any_hit = hits_df["spec_id"].nunique()
n_infeasible = hits_df[hits_df["feasibility"] == "infeasible"]["spec_id"].nunique()
# Spectra where ALL hits are feasible (no definite contaminant signal)
per_spec_feas = hits_df.groupby("spec_id")["feasibility"].apply(
    lambda x: (x == "feasible").all()
)
n_feasible_only = per_spec_feas.sum()

print(f"Total NIST23 spectra (P+N):      {total_spectra:,}")
print(f"  Positive mode:                 {total_pos:,}")
print(f"  Negative mode:                 {total_neg:,}")
print()
print("Stage 1 — mass match (per-instrument tolerance: FT=10ppm, QTOF=20ppm, IT=0.3Da):")
print(f"  Spectra with ≥1 UWPR hit:      {n_any_hit:,} ({100*n_any_hit/total_spectra:.1f}%)")
print(f"  Hit (peak, cont_mz) pairs:     {len(hits_df):,}")
print()
print("Stage 2 — formula feasibility breakdown:")
print(hits_df["feasibility"].value_counts().to_string())
print()
print(f"  Spectra with ≥1 infeasible hit (contaminant candidate):")
print(f"    {n_infeasible:,} ({100*n_infeasible/total_spectra:.1f}% of all P/N spectra)")
print(f"  Spectra where all hits are feasible (could be real peaks):")
print(f"    {n_feasible_only:,} ({100*n_feasible_only/total_spectra:.1f}% of all P/N spectra)")

Total NIST23 spectra (P+N):      1,148,062
  Positive mode:                 846,400
  Negative mode:                 301,662

Stage 1 — mass match (per-instrument tolerance: FT=10ppm, QTOF=20ppm, IT=0.3Da):
  Spectra with ≥1 UWPR hit:      345,447 (30.1%)
  Hit (peak, cont_mz) pairs:     1,173,926

Stage 2 — formula feasibility breakdown:
feasibility
feasible           648107
infeasible         508531
no_cont_formula     10324
parse_error          6964

  Spectra with ≥1 infeasible hit (contaminant candidate):
    147,620 (12.9% of all P/N spectra)
  Spectra where all hits are feasible (could be real peaks):
    195,751 (17.1% of all P/N spectra)


In [14]:
# Breakdown by instrument type
print("Spectra with ≥1 infeasible UWPR hit, by instrument type:")
infeasible_spec_ids = hits_df[hits_df["feasibility"] == "infeasible"]["spec_id"].unique()
inst_contam = spec_pn[spec_pn["spec_id"].isin(infeasible_spec_ids)]["inst_type"].value_counts()
inst_total = spec_pn["inst_type"].value_counts()
breakdown = pd.DataFrame({"contam_spectra": inst_contam, "total_spectra": inst_total}).fillna(0)
breakdown["pct"] = (100 * breakdown["contam_spectra"] / breakdown["total_spectra"]).round(1)
print(breakdown.sort_values("pct", ascending=False).to_string())

Spectra with ≥1 infeasible UWPR hit, by instrument type:
           contam_spectra  total_spectra   pct
inst_type                                     
Other                8958          16293  55.0
QQQ                  1110           2030  54.7
IT                   3292           6921  47.6
QTOF                15786          92684  17.0
FT                 118474        1029998  11.5


In [15]:
# Top contaminant m/z values by frequency (all feasibility)
print("Top 20 most frequent UWPR mass matches:")
top_cont = (
    hits_df.groupby(["cont_mz", "cont_compound", "cont_type", "feasibility"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
print(top_cont.to_string(index=False))

Top 20 most frequent UWPR mass matches:
   cont_mz                cont_compound   cont_type feasibility  count
 59.013850                   (CH3CO2)-      unknown    feasible  36488
163.038971           Dimethyl phthalate     unknown    feasible  29011
 79.021212                         DMSO     unknown  infeasible  17988
183.080441                          DPK     unknown    feasible  16752
 83.060374                 Acetonitrile     unknown    feasible  16625
100.075690       N-Methyl 2-pyrrolidone     Solvent    feasible  14303
149.023320           Phthalic anhydride Plasticiser    feasible  12407
 79.957370                        SO3-      unknown  infeasible  12085
167.033886                Phthalic acid     unknown    feasible  11335
 74.060040    Acetonitrile.1.Methanol.1     Solvent    feasible  10614
100.075690       N-Methyl 2-pyrrolidone     Solvent  infeasible  10200
 78.959060                        PO3-      unknown    feasible  10165
195.065186                          D

In [16]:
# Top infeasible hits — strongest contaminant candidates
print("Top 20 infeasible (likely contaminant) UWPR peaks:")
top_infeasible = (
    hits_df[hits_df["feasibility"] == "infeasible"]
    .groupby(["cont_mz", "cont_compound", "cont_formula", "cont_type"])
    .agg(n_spectra=("spec_id", "nunique"))
    .reset_index()
    .sort_values("n_spectra", ascending=False)
    .head(20)
)
print(top_infeasible.to_string(index=False))

Top 20 infeasible (likely contaminant) UWPR peaks:
   cont_mz             cont_compound cont_formula   cont_type  n_spectra
 79.021212                      DMSO       C2H6OS     unknown      17228
 79.957370                     SO3-         H2SO4     unknown      12038
100.075690    N-Methyl 2-pyrrolidone       C5H9NO     Solvent       9867
 96.960110                     HSO4-        H2SO4     unknown       8768
267.171973                       TBP    C12H27O4P     unknown       7900
231.116175                     n-BBS   C10H15NO2S     unknown       7393
101.003156                      DMSO       C2H6OS     unknown       6944
146.068867                       ACN        CH3CN     unknown       5610
123.000000     Dimethylaminopyridine      C7H10N2     Solvent       5443
267.000000         Tributylphosphate    C12H27O4P Plasticiser       4911
142.029705  Dimethylsulfoxide (DMSO)       C2H6OS     Solvent       4533
 74.060040 Acetonitrile.1.Methanol.1       C3H7NO     Solvent       4326


In [17]:
# Feasible hits — could be real fragments that coincide with contaminant masses
print("Top 20 feasible (could be real fragment) UWPR masses:")
top_feasible = (
    hits_df[hits_df["feasibility"] == "feasible"]
    .groupby(["cont_mz", "cont_compound", "cont_formula", "cont_type"])
    .agg(n_spectra=("spec_id", "nunique"))
    .reset_index()
    .sort_values("n_spectra", ascending=False)
    .head(20)
)
print(top_feasible.to_string(index=False))

Top 20 feasible (could be real fragment) UWPR masses:
   cont_mz                cont_compound             cont_formula   cont_type  n_spectra
 59.013850                   (CH3CO2)-                   CH3COOH     unknown      36326
163.038971           Dimethyl phthalate                 C10H10O4     unknown      28384
183.080441                          DPK                  C13H10O     unknown      16626
 83.060374                 Acetonitrile                    CH3CN     unknown      16036
100.075690       N-Methyl 2-pyrrolidone                   C5H9NO     Solvent      13503
149.023320           Phthalic anhydride                   C8H4O3 Plasticiser      12346
167.033886                Phthalic acid                   C8H6O4     unknown      10911
 74.060040    Acetonitrile.1.Methanol.1                   C3H7NO     Solvent      10270
 78.959060                        PO3-                     H3PO4     unknown      10049
195.065186                          DMP            C6H4[COOCH3]2  

In [18]:
# Save full results
out_path = Path("../data/umpr/nist20_uwpr_hits.csv")
hits_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to ../data/umpr/nist20_uwpr_hits.csv


## 8. Noise intensity and peak-count estimates

In [19]:
# ── totals across all P/N spectra ──────────────────────────────────────────
total_peaks_all = int(totals_df["total_peaks"].sum())
total_int_all   = float(totals_df["total_intensity"].sum())

# ── mass-matched hits (stage 1 — all hits regardless of feasibility) ───────
hits_mass = hits_df.groupby("spec_id").agg(
    matched_peaks=("peak_mz",  "count"),
    matched_int  =("peak_int", "sum"),
).reset_index()

total_matched_peaks = int(hits_mass["matched_peaks"].sum())
total_matched_int   = float(hits_mass["matched_int"].sum())

# ── infeasible-only hits (stage 2 — strong contaminant candidates) ─────────
hits_infeas = hits_df[hits_df["feasibility"] == "infeasible"].groupby("spec_id").agg(
    infeas_peaks=("peak_mz",  "count"),
    infeas_int  =("peak_int", "sum"),
).reset_index()

total_infeas_peaks = int(hits_infeas["infeas_peaks"].sum())
total_infeas_int   = float(hits_infeas["infeas_int"].sum())

print("=" * 60)
print("Noise estimates (across all P/N spectra)")
print("=" * 60)
print(f"  Total peaks (all P/N):              {total_peaks_all:>12,}")
print(f"  Total intensity (all P/N):          {total_int_all:>15,.0f}")
print()
print("── Stage 1: mass-matched peaks (any contaminant m/z) ──")
print(f"  Matched peaks:     {total_matched_peaks:>10,}  ({100*total_matched_peaks/total_peaks_all:.2f}% of all peaks)")
print(f"  Matched intensity: {total_matched_int:>15,.0f}  ({100*total_matched_int/total_int_all:.2f}% of total intensity)")
print()
print("── Stage 2: infeasible-only (likely contaminant) ──────")
print(f"  Infeasible peaks:  {total_infeas_peaks:>10,}  ({100*total_infeas_peaks/total_peaks_all:.2f}% of all peaks)")
print(f"  Infeasible intens: {total_infeas_int:>15,.0f}  ({100*total_infeas_int/total_int_all:.2f}% of total intensity)")


Noise estimates (across all P/N spectra)
  Total peaks (all P/N):                68,040,547
  Total intensity (all P/N):            3,303,342,348

── Stage 1: mass-matched peaks (any contaminant m/z) ──
  Matched peaks:      1,173,926  (1.73% of all peaks)
  Matched intensity:      54,957,700  (1.66% of total intensity)

── Stage 2: infeasible-only (likely contaminant) ──────
  Infeasible peaks:     508,531  (0.75% of all peaks)
  Infeasible intens:       9,014,793  (0.27% of total intensity)


In [20]:
# Per-spectrum noise fraction distributions
spec_stats = totals_df.merge(
    hits_mass.rename(columns={"matched_peaks": "mass_hit_peaks", "matched_int": "mass_hit_int"}),
    on="spec_id", how="left"
).merge(
    hits_infeas.rename(columns={"infeas_peaks": "infeas_hit_peaks", "infeas_int": "infeas_hit_int"}),
    on="spec_id", how="left"
).fillna(0)

spec_stats["mass_hit_pct_peaks"] = 100 * spec_stats["mass_hit_peaks"] / spec_stats["total_peaks"]
spec_stats["mass_hit_pct_int"]   = 100 * spec_stats["mass_hit_int"]   / spec_stats["total_intensity"]
spec_stats["infeas_pct_peaks"]   = 100 * spec_stats["infeas_hit_peaks"] / spec_stats["total_peaks"]
spec_stats["infeas_pct_int"]     = 100 * spec_stats["infeas_hit_int"]   / spec_stats["total_intensity"]

print("Per-spectrum noise fraction (mass-matched, stage 1):")
print(spec_stats[["mass_hit_pct_peaks", "mass_hit_pct_int"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(3).to_string())
print()
print("Per-spectrum noise fraction (infeasible only, stage 2):")
print(spec_stats[["infeas_pct_peaks", "infeas_pct_int"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(3).to_string())


Per-spectrum noise fraction (mass-matched, stage 1):
       mass_hit_pct_peaks  mass_hit_pct_int
count         1148037.000       1148037.000
mean                1.963             2.532
std                 6.192            10.972
min                 0.000             0.000
25%                 0.000             0.000
50%                 0.000             0.000
75%                 1.250             0.159
90%                 5.263             2.739
95%                10.000            11.664
99%                30.000            70.060
max               100.000           100.000

Per-spectrum noise fraction (infeasible only, stage 2):
       infeas_pct_peaks  infeas_pct_int
count       1148037.000     1148037.000
mean              0.609           0.530
std               3.289           4.713
min               0.000           0.000
25%               0.000           0.000
50%               0.000           0.000
75%               0.000           0.000
90%               0.842           0.118
95